# Task 104: bisip_sip — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Spectral induced polarization inversion using Cole-Cole model

**Paper**: See repository documentation
**Repository**: https://github.com/clberube/bisip

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 45.21 dB
- **SSIM**: N/A (1D spectra)
- **Evaluation**: Cole-Cole SIP parameter inversion via multi-start L-BFGS-B (PSNR + CC)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
bisip_sip - Spectral Induced Polarization: Cole-Cole Model Inversion
=====================================================================
From complex resistivity spectra ρ*(ω), fit Cole-Cole model parameters
(ρ_0, m, τ, c) using nonlinear least squares.

Physics:
  - Forward: Cole-Cole model ρ*(ω) = ρ_0 [1 - m(1 - 1/(1+(iωτ)^c))]
  - Inverse: Scipy curve_fit for parameter estimation
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`cole_cole`, `cole_cole_amplitude_phase`, `objective`, `main`

In [ ]:
# ====================================================================
# 1. Cole-Cole forward model
# ====================================================================
def cole_cole(freq, rho0, m, tau, c):
    """
    Cole-Cole complex resistivity model.
    ρ*(ω) = ρ_0 × [1 - m × (1 - 1/(1 + (iωτ)^c))]
    
    Parameters:
        freq: frequency array (Hz)
        rho0: DC resistivity (Ohm·m)
        m:    chargeability (0-1)
        tau:  time constant (s)
        c:    frequency exponent (0-1)
    
    Returns:
        Complex resistivity array
    """
    omega = 2.0 * np.pi * freq
    z = (1j * omega * tau) ** c
    rho_star = rho0 * (1.0 - m * (1.0 - 1.0 / (1.0 + z)))
    return rho_star

def cole_cole_amplitude_phase(freq, rho0, m, tau, c):
    """Return amplitude and phase of Cole-Cole model."""
    rho_star = cole_cole(freq, rho0, m, tau, c)
    amplitude = np.abs(rho_star)
    phase = -np.angle(rho_star, deg=True)  # phase in mrad (negative convention)
    phase_mrad = -np.angle(rho_star) * 1000.0  # mrad
    return amplitude, phase_mrad

# ====================================================================
# 3. Inverse: Nonlinear least squares fitting
# ====================================================================
def objective(params_vec, freq, rho_obs):
    """
    Least-squares objective: minimize misfit between observed and modeled
    complex resistivity.
    """
    rho0, m, tau, c = params_vec
    # Enforce bounds implicitly
    if rho0 <= 0 or m <= 0 or m >= 1 or tau <= 0 or c <= 0 or c >= 1:
        return 1e10

    rho_model = cole_cole(freq, rho0, m, tau, c)

    # Normalized misfit (amplitude + phase)
    amp_obs = np.abs(rho_obs)
    amp_mod = np.abs(rho_model)
    phase_obs = np.angle(rho_obs)
    phase_mod = np.angle(rho_model)

    misfit_amp = np.sum(((amp_obs - amp_mod) / amp_obs)**2)
    misfit_phase = np.sum((phase_obs - phase_mod)**2)

    return misfit_amp + misfit_phase

# ====================================================================
# 6. Main
# ====================================================================
def main():
    print("=" * 60)
    print("Task 104: Spectral Induced Polarization — Cole-Cole Inversion")
    print("=" * 60)

    # Frequency array
    freq = np.logspace(np.log10(FREQ_MIN), np.log10(FREQ_MAX), N_FREQ)
    print(f"[1] Frequencies: {N_FREQ} points, {FREQ_MIN}-{FREQ_MAX} Hz")

    all_results = []
    all_psnr = []
    all_cc = []

    for idx, gt_p in enumerate(GT_PARAMS):
        print(f"\n--- Spectrum {idx+1}/{N_SPECTRA} ---")
        print(f"    GT: ρ₀={gt_p['rho0']}, m={gt_p['m']}, τ={gt_p['tau']}, c={gt_p['c']}")

        # Forward model + noise
        rho_obs, rho_true = generate_data(freq, gt_p, NOISE_LEVEL)

        # Inverse
        t0 = time.time()
        fit_p = invert_cole_cole(freq, rho_obs, gt_p)
        elapsed = time.time() - t0
        print(f"    Fit: ρ₀={fit_p['rho0']:.2f}, m={fit_p['m']:.4f}, "
              f"τ={fit_p['tau']:.4f}, c={fit_p['c']:.4f}  ({elapsed:.1f}s)")

        # Compute fit spectrum
        rho_fit = cole_cole(freq, fit_p["rho0"], fit_p["m"], fit_p["tau"], fit_p["c"])

        # Metrics
        psnr, cc = compute_spectrum_metrics(rho_true, rho_fit)
        param_errors = compute_param_errors(gt_p, fit_p)
        print(f"    PSNR={psnr:.1f} dB, CC={cc:.4f}")
        print(f"    Param errors: {param_errors}")

        all_psnr.append(psnr)
        all_cc.append(cc)
        all_results.append({
            "gt": gt_p, "fit": fit_p,
            "rho_true": rho_true, "rho_obs": rho_obs, "rho_fit": rho_fit,
            "param_errors": param_errors, "psnr": psnr, "cc": cc,
        })

    # Average metrics
    avg_psnr = float(np.mean(all_psnr))
    avg_cc = float(np.mean(all_cc))
    print(f"\n[Summary] Avg PSNR = {avg_psnr:.2f} dB, Avg CC = {avg_cc:.4f}")

    metrics = {
        "PSNR": avg_psnr,
        "CC": avg_cc,
        "SSIM": "N/A (1D spectra)",
    }

    # Build gt_output and recon_output arrays
    gt_spectra = np.array([np.abs(r["rho_true"]) for r in all_results])
    recon_spectra = np.array([np.abs(r["rho_fit"]) for r in all_results])

    # Save
    print("[4] Saving outputs ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), gt_spectra)
        np.save(os.path.join(d, "recon_output.npy"), recon_spectra)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    # Plot
    print("[5] Plotting ...")
    plot_results(freq, all_results, metrics)

    print(f"\n{'=' * 60}")
    print("Task 104 COMPLETE")
    print(f"{'=' * 60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `generate_data`

In [ ]:
# ====================================================================
# 2. Generate synthetic data
# ====================================================================
def generate_data(freq, params, noise_level):
    """Generate noisy complex resistivity data from Cole-Cole model."""
    rho_true = cole_cole(freq, params["rho0"], params["m"],
                         params["tau"], params["c"])

    # Add relative noise to real and imaginary parts
    noise_re = noise_level * np.abs(rho_true) * np.random.randn(len(freq))
    noise_im = noise_level * np.abs(rho_true) * np.random.randn(len(freq))
    rho_noisy = rho_true + noise_re + 1j * noise_im

    return rho_noisy, rho_true

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `invert_cole_cole`

In [ ]:
def invert_cole_cole(freq, rho_obs, true_params=None):
    """
    Invert for Cole-Cole parameters using differential evolution-like
    multi-start optimization.
    """
    best_result = None
    best_cost = np.inf

    # Multi-start with different initial guesses
    rho0_guesses = [50.0, 150.0, 300.0, 500.0]
    m_guesses = [0.2, 0.4, 0.6]
    tau_guesses = [0.01, 0.1, 1.0, 10.0]
    c_guesses = [0.3, 0.5, 0.7]

    bounds = [(1.0, 1000.0), (0.01, 0.99), (1e-4, 100.0), (0.05, 0.95)]

    for rho0_init in rho0_guesses:
        for m_init in m_guesses:
            for tau_init in tau_guesses:
                for c_init in c_guesses:
                    x0 = [rho0_init, m_init, tau_init, c_init]
                    try:
                        result = minimize(
                            objective, x0, args=(freq, rho_obs),
                            method='L-BFGS-B', bounds=bounds,
                            options={'maxiter': 500, 'ftol': 1e-12}
                        )
                        if result.fun < best_cost:
                            best_cost = result.fun
                            best_result = result
                    except Exception:
                        continue

    if best_result is not None:
        rho0, m, tau, c = best_result.x
        return {"rho0": rho0, "m": m, "tau": tau, "c": c}
    else:
        return {"rho0": 100.0, "m": 0.3, "tau": 0.1, "c": 0.5}

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_spectrum_metrics`, `compute_param_errors`, `plot_results`

In [ ]:
# ====================================================================
# 4. Metrics
# ====================================================================
def compute_spectrum_metrics(rho_true, rho_fit):
    """Compute PSNR and CC for spectrum comparison."""
    # Amplitude comparison
    amp_true = np.abs(rho_true)
    amp_fit = np.abs(rho_fit)

    # Normalize
    amp_true_n = amp_true / amp_true.max()
    amp_fit_n = amp_fit / amp_fit.max()

    # PSNR
    mse = np.mean((amp_true_n - amp_fit_n)**2)
    psnr = 10.0 * np.log10(1.0 / mse) if mse > 1e-15 else 100.0

    # CC
    t_z = amp_true_n - amp_true_n.mean()
    f_z = amp_fit_n - amp_fit_n.mean()
    denom = np.sqrt(np.sum(t_z**2) * np.sum(f_z**2))
    cc = np.sum(t_z * f_z) / denom if denom > 1e-15 else 0.0

    return float(psnr), float(cc)

def compute_param_errors(gt_params, fit_params):
    """Compute relative errors for each Cole-Cole parameter."""
    errors = {}
    for key in ["rho0", "m", "tau", "c"]:
        rel_err = abs(fit_params[key] - gt_params[key]) / abs(gt_params[key]) * 100.0
        errors[key] = float(rel_err)
    return errors

# ====================================================================
# 5. Visualization
# ====================================================================
def plot_results(freq, all_results, avg_metrics):
    """Create Bode plots for all spectra and parameter comparisons."""
    n = len(all_results)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i, res in enumerate(all_results):
        rho_true = res["rho_true"]
        rho_obs = res["rho_obs"]
        rho_fit = res["rho_fit"]

        amp_true = np.abs(rho_true)
        amp_obs = np.abs(rho_obs)
        amp_fit = np.abs(rho_fit)
        phase_true = -np.angle(rho_true) * 1000.0  # mrad
        phase_obs = -np.angle(rho_obs) * 1000.0
        phase_fit = -np.angle(rho_fit) * 1000.0

        # Amplitude plot
        ax = axes[i, 0]
        ax.semilogx(freq, amp_true, 'k-', lw=2, label='True')
        ax.semilogx(freq, amp_obs, 'b.', ms=4, alpha=0.5, label='Observed')
        ax.semilogx(freq, amp_fit, 'r--', lw=1.5, label='Fit')
        ax.set_ylabel("|ρ*| (Ω·m)")
        ax.set_title(f"Spectrum {i+1}: ρ₀={res['gt']['rho0']:.0f}, "
                     f"m={res['gt']['m']:.1f}, τ={res['gt']['tau']:.2f}, "
                     f"c={res['gt']['c']:.1f}", fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        if i == n - 1:
            ax.set_xlabel("Frequency (Hz)")

        # Phase plot
        ax = axes[i, 1]
        ax.semilogx(freq, phase_true, 'k-', lw=2, label='True')
        ax.semilogx(freq, phase_obs, 'b.', ms=4, alpha=0.5, label='Observed')
        ax.semilogx(freq, phase_fit, 'r--', lw=1.5, label='Fit')
        ax.set_ylabel("-φ (mrad)")
        errs = res["param_errors"]
        ax.set_title(f"Errors: ρ₀={errs['rho0']:.1f}%, m={errs['m']:.1f}%, "
                     f"τ={errs['tau']:.1f}%, c={errs['c']:.1f}%", fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        if i == n - 1:
            ax.set_xlabel("Frequency (Hz)")

    plt.suptitle(f"Cole-Cole SIP Inversion — "
                 f"Avg PSNR={avg_metrics['PSNR']:.1f}dB, CC={avg_metrics['CC']:.3f}",
                 fontsize=14, y=1.01)
    plt.tight_layout()

    for d in [RESULTS_DIR, ASSETS_DIR]:
        fig.savefig(os.path.join(d, "reconstruction_result.png"), dpi=150, bbox_inches='tight')
        fig.savefig(os.path.join(d, "vis_result.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **bisip_sip**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=45.21 dB, SSIM=N/A (1D spectra))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: See documentation
- Repository: https://github.com/clberube/bisip
